# Session 3 — Data Structures & Functions
### Block B · Saint Mary's General Hospital — Operations Analytics Unit

> *"How would you describe a single hospital admission to a database, in plain Python?"*

**Today we move from typing values to organising values.** A patient is not one number — it is a record. A ward is not one entry — it is a list of records. The way we *shape* data decides what we can later ask of it.

By the end of this session you can:
1. Choose between **list, tuple, dict, set** with reasons.
2. Iterate efficiently and read a list comprehension out loud.
3. Write a **function** with a clear signature and a single return value.
4. Spot the most common traps: mutable defaults, scope, and silent `None`.

---

**Setting:** You just joined the Operations Analytics Unit at Saint Mary's. The CMO walked in: *"Tell me how full our wards are. By tomorrow."* Today we build the data shape that lets us answer her.

## §1 — Why structures matter

A hospital admission has many facets:
- **Who** is the patient (id, age, gender, insurance)?
- **Where** are they (ward)?
- **When** did they arrive and leave?
- **Why** were they admitted (DRG code)?
- **How much** did it cost (€)?

We need a **shape** that holds all of this together. Let's compare four shapes Python gives us.

In [ ]:
# A naive first try — parallel lists. Don't do this in real life.
patient_ids   = ["P0001", "P0002", "P0003"]
ages          = [67, 54, 82]
wards         = ["Cardiology", "InternalMedicine", "Geriatrics"]
los_days      = [5, 3, 9]

# To find the age of patient P0002 we need to *count*:
i = patient_ids.index("P0002")
print(ages[i])  # 54

# This works, but the meaning is hidden in the position.
# If we add a new column, every other list needs to stay in sync. Fragile.

## §2 — List & Tuple: ordered, indexed

A **list** is an ordered, mutable collection. We write it with `[]`.
A **tuple** is an ordered, *immutable* collection. We write it with `()`.

> Mental model: a **list** is a queue you can edit. A **tuple** is a sealed envelope — once sent, the order and content are fixed.

In [ ]:
# A list of admission dates we expect to grow during the day:
admissions_today = ["09:14", "10:02", "10:47"]
admissions_today.append("11:23")    # mutable: we add to it
print(admissions_today)

# A tuple for something that should not change — e.g. a hospital location:
saint_marys_location = (52.3759, 9.7320)   # (latitude, longitude)
# saint_marys_location[0] = 53.0  # TypeError: 'tuple' object does not support item assignment

# Why use a tuple at all? Two reasons:
# 1. It signals intent: 'this is a fixed group'.
# 2. Tuples are hashable — they can be dict keys or set elements. Lists cannot.

## §3 — Dict: keyed access, the natural shape of a record

A **dict** maps *keys* (meaning) to *values* (data). We write it with `{}`.

This is the most important data structure in clinical/operational data. A single hospital encounter naturally lives as one dict.

In [ ]:
# One encounter — clearly readable:
encounter = {
    "encounter_id": "E00042",
    "patient_id":   "P0001",
    "admission_date": "2026-01-14",
    "discharge_date": "2026-01-19",
    "ward":         "Cardiology",
    "drg_code":     "F62B",
    "los_days":     5,
    "actual_cost_eur": 3450.00,
}

# Access by key — meaning, not position:
print(encounter["ward"])             # 'Cardiology'
print(encounter["los_days"] * 220)   # rough cost estimate

In [ ]:
# Dicts are mutable. Add fields as you learn more:
encounter["readmitted_30d"] = 0
encounter["primary_nurse"] = "Schmidt"

# .get() returns None (or a fallback) if the key is absent — safer than [] for missing keys:
print(encounter.get("attending_physician", "not assigned yet"))

## §4 — Set: uniqueness, fast membership

A **set** is an unordered collection of unique, hashable elements. We write it with `{}` (without keys) or `set()`.

> Mental model: a set is what you'd write on a whiteboard when you only care *whether* something is on the list, not how often.

In [ ]:
# All distinct wards we admitted to today:
wards_today = ["Cardiology", "ICU", "Cardiology", "Surgery", "ICU", "Cardiology"]
distinct_wards = set(wards_today)
print(distinct_wards)              # {'Surgery', 'ICU', 'Cardiology'}
print(len(distinct_wards))         # 3

# Fast membership test:
print("ICU" in distinct_wards)     # True

# Set operations: which wards did we use today AND yesterday?
yesterday = {"ICU", "Geriatrics", "Pulmonology"}
both_days = distinct_wards & yesterday        # intersection
only_today = distinct_wards - yesterday       # difference
print("both:", both_days, "| only today:", only_today)

### Decision rule (memorise this)

| You need... | Use |
|---|---|
| ordered, can grow/change | **list** |
| ordered, will never change, can be a key | **tuple** |
| named fields per record | **dict** |
| unique elements, fast 'is it in?' | **set** |

## §5 — Iteration & list comprehensions

Once data lives in a structure, we walk through it. Python rewards us when we say *what* we want instead of *how* to loop.

In [ ]:
# Five encounters as a list of dicts — the natural shape of operational data:
encounters = [
    {"encounter_id": "E00001", "ward": "Cardiology",       "los_days": 5, "actual_cost_eur": 3450.00},
    {"encounter_id": "E00002", "ward": "InternalMedicine", "los_days": 3, "actual_cost_eur": 1820.00},
    {"encounter_id": "E00003", "ward": "Geriatrics",       "los_days": 9, "actual_cost_eur": 8950.00},
    {"encounter_id": "E00004", "ward": "Cardiology",       "los_days": 4, "actual_cost_eur": 2380.00},
    {"encounter_id": "E00005", "ward": "ICU",              "los_days": 7, "actual_cost_eur": 7800.00},
]

# Classic loop:
total = 0
for e in encounters:
    total += e["actual_cost_eur"]
print("Total cost:", total)

In [ ]:
# Same thing as a list comprehension — read it out loud:
# 'for every encounter e in encounters, take e["actual_cost_eur"]'
costs = [e["actual_cost_eur"] for e in encounters]
print(costs)
print("Total cost:", sum(costs))

# With a filter:
cardio_costs = [e["actual_cost_eur"] for e in encounters if e["ward"] == "Cardiology"]
print("Cardiology only:", cardio_costs)

In [ ]:
# Dict comprehension — total cost per ward:
ward_totals = {}
for e in encounters:
    ward_totals[e["ward"]] = ward_totals.get(e["ward"], 0) + e["actual_cost_eur"]
print(ward_totals)

# Comprehension form gets clunky here — old-fashioned loop is fine.
# We will see groupby() in Pandas (session 5) which makes this trivial.

## ⏸️ Mini-Challenge 1 — Build a patient roster (25 min)

Open `challenges/ch1_patient_roster.ipynb` and follow the instructions there.

**You will:** build a list-of-dicts from scratch, filter for elderly patients, compute mean age. *Fast-Track:* sort by length of stay descending.

## §6 — Functions: the smallest reusable unit of clinical reasoning

A function is named code. It takes inputs (parameters), does something, returns one value.

> Mental model: a function is a *reusable rule*. "Given a list of encounters, return the average length of stay." Once written, you trust it and reuse it.

In [ ]:
# Without a function — every time we want the mean LOS, we re-write the loop:
total = 0
for e in encounters:
    total += e["los_days"]
mean_los = total / len(encounters)
print(mean_los)

In [ ]:
# With a function — write the rule once, name it, reuse it:
def mean_los(encounter_list):
    """Return the mean length of stay across a list of encounters."""
    if not encounter_list:
        return 0
    total = sum(e["los_days"] for e in encounter_list)
    return total / len(encounter_list)

print(mean_los(encounters))

# Now we can reuse it on any subset:
cardio = [e for e in encounters if e["ward"] == "Cardiology"]
print("Mean LOS Cardiology:", mean_los(cardio))

## §7 — Scope, return, defaults, lambda

Three traps most beginners hit. Watch closely.

In [ ]:
# Trap 1: scope. Variables defined inside a function are local to it.
def add_cost(encounter, factor):
    new_cost = encounter["actual_cost_eur"] * factor
    return new_cost

scaled = add_cost(encounters[0], 1.1)
print(scaled)
# print(new_cost)   # NameError: 'new_cost' is not defined OUTSIDE the function

In [ ]:
# Trap 2: silent None.
def discharge_summary(encounter):
    print(f"Patient {encounter['encounter_id']} discharged after {encounter['los_days']} days.")
    # No return statement — Python returns None implicitly.

result = discharge_summary(encounters[0])
print("Result:", result)   # None.  This bites you in pipelines.

In [ ]:
# Trap 3: mutable default argument — a Python classic.
def bad_log(message, log=[]):
    log.append(message)
    return log

print(bad_log("admit P0001"))     # ['admit P0001']
print(bad_log("admit P0002"))     # ['admit P0001', 'admit P0002']  <-- shared!
print(bad_log("admit P0003"))     # ['admit P0001', 'admit P0002', 'admit P0003']  <-- still shared!

# The fix: default to None, create the list inside.
def good_log(message, log=None):
    if log is None:
        log = []
    log.append(message)
    return log

print(good_log("admit P0001"))    # ['admit P0001']
print(good_log("admit P0002"))    # ['admit P0002']  <-- fresh list every call

In [ ]:
# Lambda — a tiny, anonymous function. Use only when small and used in one place.
encounters_sorted = sorted(encounters, key=lambda e: e["los_days"], reverse=True)
for e in encounters_sorted:
    print(e["encounter_id"], e["los_days"])

# Rule of thumb: if your lambda has any logic beyond 'pick this field', use a real def.

## ⏸️ Mini-Challenge 2 — Filter & aggregate (25 min)

Open `challenges/ch2_filter_aggregate.ipynb`. The CFO needs a number, fast.

**You will:** filter a list of 50 encounters, sum costs, count cases. *Fast-Track:* group by DRG and return `{drg: total_cost}`.

## §8 — Klinik: Bed Occupancy Calculator (30 min, end-to-end)

The CMO wants a number: **what was our average bed occupancy on a given day, by ward?**

Logic:
1. For a given date, find all encounters where `admission_date ≤ date < discharge_date`.
2. Group those encounters by ward — count = number of beds used that day.
3. Divide by ward capacity → occupancy rate.

We will write this together as **three small functions**, each with a single job.

In [ ]:
# Ward capacities (in real life: from a config file)
WARD_CAPACITY = {
    "ICU":              16,
    "InternalMedicine": 60,
    "Surgery":          40,
    "Cardiology":       30,
    "Geriatrics":       28,
    "Pulmonology":      20,
    "Neurology":        18,
}

# A small slice of encounters for today's demo (in S4 we will load these from CSV):
demo_encounters = [
    {"encounter_id": "E00001", "ward": "Cardiology",       "admission_date": "2026-01-12", "discharge_date": "2026-01-17"},
    {"encounter_id": "E00002", "ward": "InternalMedicine", "admission_date": "2026-01-14", "discharge_date": "2026-01-17"},
    {"encounter_id": "E00003", "ward": "Geriatrics",       "admission_date": "2026-01-08", "discharge_date": "2026-01-17"},
    {"encounter_id": "E00004", "ward": "Cardiology",       "admission_date": "2026-01-15", "discharge_date": "2026-01-19"},
    {"encounter_id": "E00005", "ward": "ICU",              "admission_date": "2026-01-13", "discharge_date": "2026-01-20"},
    {"encounter_id": "E00006", "ward": "Cardiology",       "admission_date": "2026-01-16", "discharge_date": "2026-01-21"},
    {"encounter_id": "E00007", "ward": "Surgery",          "admission_date": "2026-01-15", "discharge_date": "2026-01-22"},
]

In [ ]:
from datetime import date

def is_in_house(encounter, target_date):
    """Return True if the patient was an inpatient on target_date."""
    adm = date.fromisoformat(encounter["admission_date"])
    dc  = date.fromisoformat(encounter["discharge_date"])
    return adm <= target_date < dc

# Quick test
target = date(2026, 1, 16)
print(is_in_house(demo_encounters[0], target))   # True
print(is_in_house(demo_encounters[1], target))   # False (already discharged on 17 → wait, 16 < 17 → True)

In [ ]:
def beds_used_per_ward(encounters_list, target_date):
    """Return {ward: count_of_inpatients} for target_date."""
    counts = {}
    for e in encounters_list:
        if is_in_house(e, target_date):
            counts[e["ward"]] = counts.get(e["ward"], 0) + 1
    return counts

print(beds_used_per_ward(demo_encounters, date(2026, 1, 16)))

In [ ]:
def occupancy_rate(encounters_list, target_date, capacity=WARD_CAPACITY):
    """Return {ward: occupancy_rate_0_to_1} for target_date."""
    used = beds_used_per_ward(encounters_list, target_date)
    return {ward: used.get(ward, 0) / cap for ward, cap in capacity.items()}

rates = occupancy_rate(demo_encounters, date(2026, 1, 16))
for ward, r in rates.items():
    print(f"{ward:>20s}  {r:5.0%}")

**Notice what we just did.**

We wrote three tiny functions, each doing exactly one thing:
- `is_in_house()` — boolean test for one encounter on one day
- `beds_used_per_ward()` — uses the test on a list, groups by ward
- `occupancy_rate()` — divides counts by capacity

This is **modular thinking**. When something breaks, we know exactly which function to debug. When the rule changes (e.g., we want occupancy at midnight instead of mid-day), we change one function, not the whole script.

In session 4 we will replace `demo_encounters` with the real CSV — and these three functions will keep working unchanged.

## ⏸️ Mini-Challenge 3 — Refactor spaghetti into functions (15 min)

Open `challenges/ch3_refactor.ipynb`. You'll see a 30-line procedural script. Refactor it into three named functions.

## §9 — Differential Diagnosis: the bugs we will see again

| Symptom | Likely cause | Fix |
|---|---|---|
| `TypeError: 'NoneType' is not subscriptable` | function returned None implicitly | add explicit `return` |
| Function "remembers" data from last call | mutable default argument | use `arg=None` |
| `KeyError: 'ward'` | missing field in some dict | use `.get("ward", "Unknown")` |
| List grew unexpectedly | shared reference, not a copy | `new = old.copy()` or `list(old)` |
| `NameError` outside the function | scope: variable was local | return it instead |

---

## Take-home for tomorrow

> **Pick three encounters from your own life as a "patient" of any system** (a hotel stay, a flight, a doctor's visit). Model each as a Python dict with at least 5 fields. Write one function that takes a list of these dicts and returns a single summary number. Bring the file to session 4.

## Cliffhanger → Session 4 (Thursday)

Tomorrow the CFO sends us the real CSV export from the hospital information system. We will see what it takes to load 320 real encounters from disk, deal with messy columns, and meet **NumPy and Pandas** — the two libraries every data scientist lives in.

You won't need to write your own loops anymore. The structures we built today are exactly what Pandas gives you for free.